## FIX: Variant difference owed to phenotype-imposed MAF when generating sumstats

In [ ]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
# details: https://privefl.github.io/bigsnpr-extdoc/polygenic-scores-pgs.html

# wrote a package that contain all dependencies for runnign LDpred2
# and combining various functions into easy to use pipelines. Note,
# that this will also load libraries, e.g. bigsnpr, bigassert
devtools::load_all('utils/modules/R/prstools')


get_sd_y <- function(path_cts_phenotypes, phenotype){
    stopifnot(file.exists(path_cts_phenotypes))
    phenos <- fread(path_cts_phenotypes)
    stopifnot(phenotype %in% colnames(phenos))
    y <- phenos[[phenotype]]
    return(sd(y, na.rm = TRUE))
}

In [12]:
args <- list(
    trait = "binary",
    pred = "data/prs/hapmap/ukb_500k/ukb_hapmap_500k_eur_chr21.bed",
    gwas = "data/prs/sumstat/binary/ukb_hapmap_500k_eur_NASH_combined.txt.gz",
    #gwas = "data/prs/sumstat/binary/ukb_hapmap_500k_eur_HYP_combined.txt.gz",
    #ldsc = "data/prs/ldsc/ldsc_Alanine_aminotransferase_residual.rds",
    ld_dir = "data/prs/hapmap/ld/matrix",
    ld_bed = "data/prs/hapmap/ld/unrel_kin_eur_10k/short_merged_ukb_hapmap_rand_10k_eur.bed"
)

In [13]:
# setup parallel environment
NCORES <- max(1, nb_cores())
bigparallelr::assert_cores(NCORES)

# Load LD matrix and summary statistics
gwas <- read_hail_sumstat(args$gwas, trait = args$trait)
ld_data <- load_bigsnp_from_bed(args$ld_bed)

# match summary stats and LD data
info_snp <- snp_match(gwas, ld_data$map, join_by_pos = TRUE, strand_flip = FALSE)

# qc summary statistics
if (args$trait %in% "binary"){
    qc <- qc_sumstat_binary(ld_data$G, info_snp, ncores = NCORES)
} else {
# Note, the cts traits require the standard deviation of the phenotype
    sd_y <- get_sd_y(args$path_cts_phenotypes, args$phenotype)
    qc <- qc_sumstat_cts(ld_data$G, info_snp, sd_y, ncores = NCORES)
}

1,083,344 variants to be matched.

Some duplicates were removed.

1,082,543 variants have been matched; 0 were flipped and 0 were reversed.



List of 3
 $ is_bad: logi [1:875208] FALSE FALSE FALSE FALSE TRUE FALSE ...
 $ sd_val: num [1:875208] 0.544 0.501 0.501 0.538 NA ...
 $ sd_ss : num [1:875208] 0.514 0.475 0.475 0.518 0.588 ...
